In [0]:
/*
=============================================================================
PURPOSE: Market Risk Premium Dimension Table for CAPM Calculations
=============================================================================
This view provides a reference table of market risk premium (MRP) scenarios
used in Capital Asset Pricing Model (CAPM) calculations and portfolio analysis.

TARGET SCHEMA: thekitchen.t (transformed data layer)
GRAIN: One row per MRP scenario

WHAT IS MARKET RISK PREMIUM?
The market risk premium is the additional return investors expect for taking on
the risk of investing in the stock market versus risk-free assets (e.g., Treasury bonds).
- Formula: MRP = Expected Market Return - Risk-Free Rate
- Typically ranges from 4% to 7% depending on market conditions
- Used in CAPM: Expected Return = Risk-Free Rate + Beta × MRP

BUSINESS VALUE:
- Enables scenario analysis with different market outlook assumptions
- Supports sensitivity testing: "What if markets become more volatile?"
- Provides consistent MRP values for portfolio valuation and planning
- Facilitates stress testing under various market conditions

SCENARIO DESCRIPTIONS:
- Very Low (4.0%): Very optimistic, bull market conditions
- Low (4.5%): Optimistic, stable growth environment
- Moderate (5.0%): Balanced, long-term historical average
- Base (5.5%): Standard assumption for most analyses
- Elevated (6.0%): Cautious, increased uncertainty
- High (6.5%): Pessimistic, market stress conditions
- Very High (7.0%): Crisis scenario, high volatility

USAGE PATTERN:
- Join to beta and risk-free rate tables for CAPM calculations
- Filter by MrpScenario for specific market outlook assumptions
- Use SortOrder for proper chronological/severity ordering in reports
=============================================================================
*/

CREATE OR REPLACE VIEW t.marketriskpremium_tmp1 AS
-- ============================================
-- Market Risk Premium Scenarios
-- ============================================
-- Inline table of MRP values for different market conditions
SELECT
  *
FROM
  VALUES
    -- OPTIMISTIC SCENARIOS (MRP < 5.0%)
    -- Bull market, low volatility, strong economic growth
    (1, 'MRP_040_US', 'Very Low',  'US', CAST(0.0400 AS DECIMAL(6, 4)), 'Very optimistic market assumptions'),  -- 4.0%
    (2, 'MRP_045_US', 'Low',       'US', CAST(0.0450 AS DECIMAL(6, 4)), 'Low equity risk premium'),             -- 4.5%
    
    -- MODERATE SCENARIOS (MRP = 5.0% - 5.5%)
    -- Balanced outlook, historical average conditions
    (3, 'MRP_050_US', 'Moderate',  'US', CAST(0.0500 AS DECIMAL(6, 4)), 'Moderate long-term ERP'),               -- 5.0%
    (4, 'MRP_055_US', 'Base',      'US', CAST(0.0550 AS DECIMAL(6, 4)), 'Standard base-case ERP'),               -- 5.5% (most commonly used)
    
    -- ELEVATED SCENARIOS (MRP = 6.0% - 6.5%)
    -- Increased uncertainty, market volatility, economic headwinds
    (5, 'MRP_060_US', 'Elevated',  'US', CAST(0.0600 AS DECIMAL(6, 4)), 'Elevated risk environment'),            -- 6.0%
    (6, 'MRP_065_US', 'High',      'US', CAST(0.0650 AS DECIMAL(6, 4)), 'High uncertainty / stress'),            -- 6.5%
    
    -- CRISIS SCENARIOS (MRP >= 7.0%)
    -- Market stress, recession, high volatility (2008 Financial Crisis, 2020 COVID-19)
    (7, 'MRP_070_US', 'Very High', 'US', CAST(0.0700 AS DECIMAL(6, 4)), 'Crisis-level assumptions')              -- 7.0%
    
AS v
(
  SortOrder,      -- Numeric order for sorting (1-7, low to high risk)
  MrpKey,         -- Unique identifier (e.g., "MRP_055_US")
  MrpScenario,    -- Human-readable scenario name (e.g., "Base", "High")
  Market,         -- Market/country identifier ("US" for US equities)
  MrpValue,       -- Market risk premium as decimal (e.g., 0.0550 = 5.5%)
  Description     -- Contextual description of scenario assumptions
);

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: List all MRP scenarios with percentage display
SELECT
  SortOrder,
  MrpScenario,
  ROUND(MrpValue * 100, 2) AS mrp_percentage,
  Description
FROM t.marketriskpremium_tmp1
ORDER BY SortOrder;

-- Example 2: Calculate expected returns under different MRP scenarios
-- Uses latest 10Y Treasury as risk-free rate and market beta (β=1.0)
SELECT
  m.MrpScenario,
  m.MrpValue AS market_risk_premium,
  rf.Yield AS risk_free_rate,
  (rf.Yield + (1.0 * m.MrpValue)) AS expected_market_return,
  ROUND((rf.Yield + (1.0 * m.MrpValue)) * 100, 2) AS expected_return_pct
FROM t.marketriskpremium_tmp1 m
CROSS JOIN (
  SELECT Yield FROM t.us_treasury_ten_years 
  WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_ten_years)
) rf
ORDER BY m.SortOrder;

-- Example 3: Scenario comparison - Base vs High stress
WITH scenarios AS (
  SELECT 
    MrpScenario,
    MrpValue,
    (SELECT Yield FROM t.us_treasury_ten_years 
     WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_ten_years)) AS risk_free_rate
  FROM t.marketriskpremium_tmp1
  WHERE MrpScenario IN ('Base', 'High')
)
SELECT
  MrpScenario,
  ROUND(MrpValue * 100, 2) AS mrp_pct,
  ROUND(risk_free_rate * 100, 2) AS rf_pct,
  ROUND((risk_free_rate + MrpValue) * 100, 2) AS expected_return_pct,
  ROUND((risk_free_rate + MrpValue) * 100, 2) - 
    ROUND((risk_free_rate + (SELECT MrpValue FROM t.marketriskpremium_tmp1 WHERE MrpScenario = 'Base')) * 100, 2) 
    AS diff_from_base_pct
FROM scenarios;

-- Example 4: Full CAPM matrix with all betas and MRP scenarios
SELECT
  b.BetaLabel,
  m.MrpScenario,
  ROUND(rf.Yield * 100, 2) AS risk_free_rate_pct,
  ROUND(m.MrpValue * 100, 2) AS mrp_pct,
  ROUND((rf.Yield + (b.BetaValue * m.MrpValue)) * 100, 2) AS expected_return_pct
FROM t.marketbeta_tmp1 b
CROSS JOIN t.marketriskpremium_tmp1 m
CROSS JOIN (
  SELECT Yield FROM t.us_treasury_ten_years 
  WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_ten_years)
) rf
WHERE m.MrpScenario IN ('Low', 'Base', 'High')  -- Focus on key scenarios
ORDER BY b.BetaValue, m.SortOrder;
=============================================================================
*/
